# OpenSees ML Example — Cantilever Sweep + Linear Regression

End-to-end ML workflow on DesignSafe using `dapi` and `designsafe-agnostic-app`:

1. Sweep `NodalMass × LCol × E` over a 2D cantilever pushover in OpenSeesPy.
2. Each combo runs as an independent PyLauncher task.
3. After all tasks finish, `POST_JOB_SCRIPT.sh` aggregates `out_*/metrics.json`, fits a linear regression predicting `log(period)`, and renders a diagnostic PDF/PNG.
4. Archive contains the fitted model JSON, per-task predictions CSV, a short report, and `opensees_ml_diagnostics.png`.

Mirrors the ML layer of Silvia Mazzoni's `nga_mpi_ml_example.py` (feature matrix → split → linear fit → JSON model + diagnostic figure). Because `T = 2π·√(M·L³/(3·E·I))`, the regression should recover `coef ≈ [const, 0.5, 1.5, -0.5]` with `R² ≈ 1.0`.

**Files in this folder** (already on disk; the notebook does not rewrite them):
- `cantilever.py` — one OpenSeesPy pushover task
- `aggregate_and_train.py` — aggregation + regression
- `postprocess.py` — diagnostic plots (PDF/PNG)
- `POST_JOB_SCRIPT.sh` — runs the aggregator and plotter after PyLauncher
- `PIP_INSTALLS_FILE.txt` — extra packages for the job

Run this notebook **from the folder that contains those files** (e.g. copy the whole folder into your `~/MyData/`).

In [ ]:
%pip install dapi --quiet

In [ ]:
import os
from pathlib import Path
from dapi import DSClient

ds = DSClient()
input_dir = Path(os.getcwd())
print(f"Input directory: {input_dir}")
for fn in (
    "cantilever.py",
    "aggregate_and_train.py",
    "postprocess.py",
    "POST_JOB_SCRIPT.sh",
    "PIP_INSTALLS_FILE.txt",
):
    assert (input_dir / fn).exists(), f"Missing: {fn}"
print("All required files present.")

## 1. Define the parameter sweep

5 × 5 × 3 = **75 tasks**. Increase the lists for a bigger training set.

In [ ]:
sweep = {
    "NODAL_MASS": [4.19, 4.39, 4.59, 4.79, 4.99],  # kip*s^2/in
    "LCOL": [100, 200, 300, 400, 500],  # in
    "E": [3600, 4227, 5000],  # ksi
}

command = (
    "python3 cantilever.py --NodalMass NODAL_MASS --LCol LCOL --E E "
    "--outDir out_NODAL_MASS_LCOL_E"
)

## 2. Preview

In [ ]:
preview = ds.jobs.parametric_sweep.generate(command, sweep, preview=True)
print(f"Total runs: {len(preview)}")
preview.head()

## 3. Write `runsList.txt` and `call_pylauncher.py`

In [ ]:
commands = ds.jobs.parametric_sweep.generate(command, sweep, str(input_dir))
print(f"Generated {len(commands)} task commands\n")
print("=== runsList.txt (first 5 lines) ===")
print("\n".join((input_dir / "runsList.txt").read_text().splitlines()[:5]))
print("...")

## 4. Submit the job

Set your TACC allocation, then run.

In [ ]:
allocation = "BCS20003"  # <-- replace with your allocation

job = ds.jobs.parametric_sweep.submit(
    str(input_dir),
    app_id="designsafe-agnostic-app",
    allocation=allocation,
    node_count=1,
    cores_per_node=48,
    max_minutes=30,
    queue="skx-dev",
    extra_env_vars=[
        {"key": "POST_JOB_SCRIPT", "value": "POST_JOB_SCRIPT.sh"},
        {"key": "PIP_INSTALLS_FILE", "value": "PIP_INSTALLS_FILE.txt"},
    ],
)
print(f"Job UUID: {job.uuid}")
job.monitor(interval=30)

## 5. Inspect ML outputs

Expected: `coef_log_NodalMass ≈ 0.5`, `coef_log_LCol ≈ 1.5`, `coef_log_E ≈ −0.5`, `R² ≈ 1.0`.

In [ ]:
import json

archive_uri = job.archive_uri
print(f"Archive: {archive_uri}")
for item in ds.files.list(f"{archive_uri}/inputDirectory/ml_results"):
    print(f"  {item.name}")

job.download_output(
    "inputDirectory/ml_results/opensees_ml_model.json",
    "opensees_ml_model.json",
)
with open("opensees_ml_model.json") as f:
    model = json.load(f)
print(json.dumps(model, indent=2))

## 6. Show diagnostic plots

`postprocess.py` runs on the compute node as part of `POST_JOB_SCRIPT.sh` and produces a 2×4 figure: feature histograms, target distribution, predicted-vs-truth scatter, residual panel, residual-vs-predicted, and a coefficient bar chart with the physics-expected exponents overlaid as red reference lines.

In [ ]:
from IPython.display import Image, display

job.download_output(
    "inputDirectory/ml_results/opensees_ml_diagnostics.png",
    "opensees_ml_diagnostics.png",
)
display(Image("opensees_ml_diagnostics.png"))

Also download the PDF version (vector) and the per-task predictions CSV if you want to re-plot or do further analysis.

In [ ]:
job.download_output(
    "inputDirectory/ml_results/opensees_ml_diagnostics.pdf",
    "opensees_ml_diagnostics.pdf",
)
job.download_output(
    "inputDirectory/ml_results/opensees_ml_preds.csv",
    "opensees_ml_preds.csv",
)
print("Downloaded diagnostics.pdf and preds.csv.")